In [1]:
import subprocess
from pathlib import Path

from config.config_loader import ConfigLoader, TrackerConfig
from core.adapter.predictions_adapter_factory import StreamedPredictionsAdapterFactory
from core.heatmap import SpeedHeatmapBuilder
from core.heatmap.directional.directional_heatmap_builder import (
    DirectionalHeatmapBuilder,
)
from core.heatmap.visualizer import TrackVisualizer
from core.heatmap.visualizer.heatmap_visualizer import HeatmapVisualizer
from core.helpers.camera_info import CameraInfo
from core.momentum import camera_to_world_mapper
from core.momentum.camera_to_world_mapper import CameraToWorldMapper
from core.momentum.momentum import MomentumTracker
from core.helpers import DataSourceInfoReader
from core.helpers.point import PointUtil
from models.yolo.yolo import YOLOModel
from project_root import PROJECT_ROOT
import time

# path = "data/datasets/mall_dataset/frames/"
path = PROJECT_ROOT / "data/datasets/yt/walking_people.mp4"
# path = PROJECT_ROOT / "data/datasets/yt/m6-motorway-trim.mp4"



In [2]:
def _resolve_path(image_path: str) -> Path:
    path_obj = Path(image_path)
    if path_obj.is_absolute():
        return path_obj
    return PROJECT_ROOT / path_obj


debug_path = PROJECT_ROOT / Path("out/debug")


def main() -> None:
    model = YOLOModel(path)
    raw_predictions = model.run_tracking(show=False, stream=True)
    predictions_adapter = StreamedPredictionsAdapterFactory.for_model(model)
    metadata = DataSourceInfoReader(path).read()

    if metadata is None:
        raise ValueError("Could not read metadata from source")

    assert metadata.fps is not None

    tracker_config = ConfigLoader.load_tracker_config(TrackerConfig.BOTSORT)
    directional_heatmap = (
        DirectionalHeatmapBuilder()
        .with_height(metadata.height)
        .with_width(metadata.width)
        .with_frames(metadata.frames)
        .with_fps(metadata.fps)
        .with_momentum_buffer_size(15)
        .with_max_lost_frames(tracker_config.get("track_buffer", 10))
        .build()
    )

    speed_heatmap = (
        SpeedHeatmapBuilder()
        .with_height(metadata.height)
        .with_width(metadata.width)
        .with_frames(metadata.frames)
        .with_fps(metadata.fps)
        .with_momentum_buffer_size(30)
        .with_half_life_time(4)
        .with_max_lost_frames(tracker_config.get("track_buffer", 10))
        .with_speed_max(90)
        .build()
    )

    mapper = CameraToWorldMapper(CameraInfo().get_transformation_matrix())
    momentum = MomentumTracker(metadata.fps, 15, tracker_config.get("track_buffer", 10), mapper)
    hv = TrackVisualizer()


    debug_path.mkdir(parents=True, exist_ok=True)

    for num, prediction in enumerate(raw_predictions):
        processed_prediction = predictions_adapter.to_predictions(prediction)


        clamped_points = PointUtil.clamp_points_to_heatmap_points(
            processed_prediction.points, metadata.width, metadata.height
        )
        idxs = [int(idx.track_id) for idx in processed_prediction.points]
        updates = momentum.update_batch(
            idxs, clamped_points
        )

        lost_tracks_updates = momentum.flush_lost_tracks_buffers(
            set(point.track_id for point in processed_prediction.points)
        )

        hv.draw(
            processed_prediction.image,
            dict(zip(idxs, updates)),
            save_path=debug_path / f"{num}.png",
        )



In [3]:
main()

YOLO logging enabled
Running YOLO tracking on source: /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4
Loading YOLO model: /Users/wiktor/Projects/density-methods/data/weights/yolo26n.pt



video 1/1 (frame 1/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 20 persons, 22.6ms
video 1/1 (frame 2/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 18 persons, 21.6ms
video 1/1 (frame 3/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 19 persons, 17.7ms
video 1/1 (frame 4/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 21 persons, 17.9ms
video 1/1 (frame 5/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 22 persons, 16.9ms
video 1/1 (frame 6/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 22 persons, 17.1ms
video 1/1 (frame 7/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 24 persons, 16.9ms
video 1/1 (frame 8/341) /Users/wiktor/Projects/density-methods/data/datasets/yt/walking_people.mp4: 384x640 24